# Hierarchical Medical Knowledge Graph Week Notebook

This notebook implements the 7-day workflow for creating, analyzing, and evaluating a synthetic hierarchical medical knowledge graph (KG) for the diabetes domain. Each code cell is preceded by an explanatory markdown block and includes comments at the top of the cell, following the requested layout.

Outline:
- Day 1: Create synthetic hierarchical KG + ground truth
- Day 2: Implement hierarchical community detection algorithms (Leiden, Louvain, Agglomerative)
- Day 3: Define hierarchical objective with Cohesion, Vertical Consistency, Cross-Branch Diversity
- Day 4-5: Experiments and visualizations
- Day 6: RAG evaluation with hierarchical retrieval
- Day 7: Paper outline

### Setup: Install and import dependencies

This cell installs (if needed) and imports all Python packages used across the week. It also sets global configuration, random seeds, and plotting styles.

In [ ]:
# Dependencies and environment setup
# - Installs missing packages (quiet where possible)
# - Imports libraries used in subsequent days
# - Sets seeds and plotting defaults

import sys, subprocess, importlib, os, random

# Helper to install a pip package if missing
def ensure_package(pkg, import_name=None, extras=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        to_install = pkg if extras is None else f"{pkg}[{extras}]"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", to_install])

# Core
ensure_package("networkx")
ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("scipy")

# Graph community packages
ensure_package("python-igraph", import_name="igraph")
ensure_package("leidenalg")
ensure_package("python-louvain", import_name="community")

# Embeddings and clustering
ensure_package("scikit-learn", import_name="sklearn")
ensure_package("node2vec")

# Visualization
ensure_package("seaborn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Re-import now that packages are ensured
import igraph as ig
import leidenalg
import community as community_louvain
from sklearn.cluster import AgglomerativeClustering

# Global config
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

plt.style.use("seaborn-v0_8")
sns.set_context("talk")
print("Environment ready.")

### Day 1: Create synthetic hierarchical medical KG (3 layers) and summarize

This section creates a directed hierarchical KG for the diabetes domain with 3 layers (2: categories, 1: concepts, 0: entities). It also prints basic counts per layer and stores the ground truth hierarchy for later evaluations.

In [ ]:
# Build hierarchical medical KG (directed) with 3 layers
# - Layer 2: Disease categories
# - Layer 1: Concepts/conditions
# - Layer 0: Entities/facts

import networkx as nx
import numpy as np


def create_hierarchical_medical_kg(n_layers: int = 3):
    """
    Creates a hierarchical KG for diabetes domain
    Layer 2 (top): Disease categories (4-6 nodes)
    Layer 1 (middle): Specific conditions/concepts (20-30 nodes)
    Layer 0 (bottom): Entities/facts (100-200 nodes)
    """
    hierarchy = {
        "Metabolic_Disorders": {
            "Diabetes_Type_2": {
                "entities": [
                    "Insulin_Resistance", "Hyperglycemia", "Beta_Cell_Dysfunction",
                    "Metabolic_Syndrome", "Prediabetes", "Gestational_Diabetes"
                ],
                "treatments": [
                    "Metformin", "Sulfonylureas", "GLP1_Agonists", "SGLT2_Inhibitors",
                    "Insulin_Therapy", "Lifestyle_Modification"
                ],
                "symptoms": [
                    "Polyuria", "Polydipsia", "Fatigue", "Blurred_Vision", "Weight_Loss"
                ]
            },
            "Diabetes_Type_1": {
                "entities": ["Autoimmune_Destruction", "Beta_Cell_Loss", "Ketoacidosis"],
                "treatments": ["Insulin_Pump", "Multiple_Daily_Injections", "Continuous_Glucose_Monitor"],
                "symptoms": ["Rapid_Onset", "Weight_Loss", "DKA"]
            },
            "Diabetic_Complications": {
                "entities": [
                    "Retinopathy", "Nephropathy", "Neuropathy", "Cardiovascular_Disease",
                    "Diabetic_Foot", "Stroke_Risk", "Heart_Attack_Risk"
                ]
            }
        },
        "Cardiovascular_Diseases": {
            "Hypertension": {
                "entities": ["High_BP", "Systolic_Pressure", "Diastolic_Pressure"],
                "treatments": ["ACE_Inhibitors", "Beta_Blockers", "Diuretics"]
            },
            "Atherosclerosis": {
                "entities": ["Plaque_Formation", "Arterial_Stiffness", "Endothelial_Dysfunction"]
            }
        },
        "Endocrine_Disorders": {
            "Thyroid_Disorders": {
                "entities": ["Hypothyroidism", "Hyperthyroidism", "TSH_Abnormalities"]
            },
            "Obesity": {
                "entities": ["BMI_Over_30", "Visceral_Fat", "Leptin_Resistance"],
                "link_to_diabetes": ["Insulin_Resistance", "Metabolic_Syndrome"]
            }
        }
    }

    G = nx.DiGraph()
    layer_assignments = {}

    for l2_category, l1_dict in hierarchy.items():
        G.add_node(l2_category, layer=2, type="category")
        layer_assignments[l2_category] = 2
        for l1_concept, l0_dict in l1_dict.items():
            G.add_node(l1_concept, layer=1, type="concept")
            G.add_edge(l2_category, l1_concept, relation="has_subconcept")
            layer_assignments[l1_concept] = 1
            for entity_type, entities in l0_dict.items():
                for entity in entities:
                    entity_node = f"{l1_concept}_{entity}"
                    G.add_node(entity_node, layer=0, type=entity_type, original_name=entity)
                    G.add_edge(l1_concept, entity_node, relation="has_entity")
                    layer_assignments[entity_node] = 0

    # Cross-edges for cohesion and cross-branch connections
    if G.has_node("Diabetes_Type_2_Metformin") and G.has_node("Diabetes_Type_2_Lifestyle_Modification"):
        G.add_edge(
            "Diabetes_Type_2_Metformin",
            "Diabetes_Type_2_Lifestyle_Modification",
            relation="combined_with",
            weight=0.8,
        )

    if G.has_node("Diabetes_Type_2_Insulin_Resistance") and G.has_node("Obesity_Insulin_Resistance"):
        G.add_edge(
            "Diabetes_Type_2_Insulin_Resistance",
            "Obesity_Insulin_Resistance",
            relation="shared_mechanism",
            weight=0.7,
        )

    return G, layer_assignments, hierarchy


# Generate KG and summarize
G_medical, layer_map, ground_truth_hierarchy = create_hierarchical_medical_kg()
print(f"Created hierarchical KG with {G_medical.number_of_nodes()} nodes and {G_medical.number_of_edges()} edges")
print(
    f"Layer 2 (categories): {sum(1 for n in G_medical.nodes() if layer_map[n] == 2)}\n"
    f"Layer 1 (concepts): {sum(1 for n in G_medical.nodes() if layer_map[n] == 1)}\n"
    f"Layer 0 (entities): {sum(1 for n in G_medical.nodes() if layer_map[n] == 0)}"
)

### Day 2: Hierarchical community detection algorithms

We implement three hierarchical clustering strategies:
- Hierarchical Leiden via multi-resolution
- Hierarchical Louvain using dendrogram levels
- Agglomerative clustering on node embeddings (node2vec)

In [ ]:
# Hierarchical community detection algorithms
# - Leiden at multiple resolutions
# - Louvain dendrogram levels
# - Agglomerative clustering on node2vec embeddings

import itertools
from typing import Dict, Any


def hierarchical_leiden_multilevel(G: nx.Graph, layer_assignments: Dict[str, int], resolution_sequence=[2.0, 1.0, 0.5]):
    """
    Run Leiden at multiple resolutions to create hierarchy
    High resolution → fine-grained (Layer 0)
    Low resolution → coarse (Layer 2)
    Returns: dict like {"layer_2": {node: comm_id, ...}, ...}
    """
    # Convert to igraph preserving node order for mapping
    nodes_order = list(G.nodes())
    G_ig = ig.Graph.TupleList(G.edges(), directed=G.is_directed(), vertex_name_attr="name")
    # Ensure all isolated nodes exist in igraph
    missing = set(nodes_order) - set(G_ig.vs["name"]) if G_ig.vs else set(nodes_order)
    if missing:
        for m in missing:
            G_ig.add_vertex(name=m)

    hierarchies: Dict[str, Dict[Any, int]] = {}
    for level, resolution in enumerate(resolution_sequence):
        partition = leidenalg.find_partition(
            G_ig,
            leidenalg.RBConfigurationVertexPartition,
            resolution_parameter=resolution,
        )
        membership = {v["name"]: comm for v, comm in zip(G_ig.vs, partition.membership)}
        hierarchies[f"layer_{2 - level}"] = membership
    return hierarchies


def hierarchical_louvain(G: nx.Graph):
    """Standard Louvain produces a dendrogram (hierarchy)."""
    if G.is_directed():
        G_undirected = G.to_undirected()
    else:
        G_undirected = G
    dendro = community_louvain.generate_dendrogram(G_undirected)
    hierarchies: Dict[str, Dict[Any, int]] = {}
    # Map nodes to partition at each level
    for level in range(len(dendro)):
        part = community_louvain.partition_at_level(dendro, level)
        hierarchies[f"layer_{level}"] = part
    # Normalize to 3 layers if needed (coarse at layer_2)
    # If dendro has fewer levels, upcast last available to remaining layers
    for lname in ["layer_0", "layer_1", "layer_2"]:
        if lname not in hierarchies:
            # pick closest lower level available
            fallback = max((int(k.split("_")[1]) for k in hierarchies), default=0)
            hierarchies[lname] = hierarchies.get(f"layer_{fallback}", {})
    return hierarchies


def agglomerative_clustering_on_embeddings(G: nx.Graph, n_clusters_per_level=[50, 12, 4]):
    """
    Use node2vec embeddings + hierarchical (fixed k per level)
    Returns: dict {"layer_0": ..., "layer_1": ..., "layer_2": ...}
    """
    # node2vec expects networkx Graph (undirected is typical)
    from node2vec import Node2Vec

    if isinstance(G, nx.DiGraph):
        G_emb = G.to_undirected()
    else:
        G_emb = G

    node_order = list(G_emb.nodes())
    node2vec = Node2Vec(G_emb, dimensions=64, walk_length=30, num_walks=200, seed=SEED, workers=1, quiet=True)
    model = node2vec.fit(window=10, min_count=1, batch_words=64)

    # Ensure consistent key type (string)
    embeddings = np.array([model.wv[str(n)] if str(n) in model.wv else np.zeros(64) for n in node_order])

    hierarchies: Dict[str, Dict[Any, int]] = {}
    for level, n_clusters in enumerate(n_clusters_per_level):
        clustering = AgglomerativeClustering(n_clusters=n_clusters)
        labels = clustering.fit_predict(embeddings)
        hierarchies[f"layer_{2 - level}"] = {node: int(label) for node, label in zip(node_order, labels)}
    return hierarchies


### Day 3: Hierarchical objective functions (Cohesion, Vertical Consistency, Cross-Branch Diversity)

This section implements the core novelty: a balanced objective combining horizontal cohesion, vertical consistency, and cross-branch diversity. Each component is computed per the provided definitions.

In [ ]:
# Hierarchical objective and metrics
# - Cohesion per layer and average
# - Vertical consistency across parent-child
# - Cross-branch diversity at a given layer

from typing import Mapping


def hierarchical_cohesion(G: nx.Graph, hierarchies: Mapping[str, Mapping[Any, int]], layer_assignments: Mapping[Any, int]):
    """
    Measures cohesion at EACH layer separately
    Returns: {layer_0: score, layer_1: score, layer_2: score}
    """
    cohesion_per_layer: Dict[str, float] = {}
    for layer_name, communities in hierarchies.items():
        try:
            layer_num = int(layer_name.split("_")[1])
        except Exception:
            continue
        nodes_in_layer = [n for n, l in layer_assignments.items() if l == layer_num]
        if not nodes_in_layer:
            cohesion_per_layer[layer_name] = 0.0
            continue
        layer_cohesion = 0.0
        comm_ids = set(communities.get(n, -1) for n in nodes_in_layer)
        comm_ids.discard(-1)
        if not comm_ids:
            cohesion_per_layer[layer_name] = 0.0
            continue
        for comm_id in comm_ids:
            nodes_in_comm = [n for n in nodes_in_layer if communities.get(n, -1) == comm_id]
            if len(nodes_in_comm) < 2:
                continue
            subgraph = G.subgraph(nodes_in_comm)
            density = nx.density(subgraph)
            layer_cohesion += float(density)
        cohesion_per_layer[layer_name] = layer_cohesion / max(len(comm_ids), 1)
    return cohesion_per_layer


def vertical_consistency(G: nx.Graph, hierarchies: Mapping[str, Mapping[Any, int]], layer_assignments: Mapping[Any, int]):
    """
    NEW METRIC: Children should belong to semantically related parents
    Measures if Layer 0 entities in same Layer 1 cluster are truly related
    """
    if "layer_0" not in hierarchies or "layer_1" not in hierarchies:
        return 0.0
    layer0_comms = hierarchies["layer_0"]
    layer1_comms = hierarchies["layer_1"]
    consistency_scores: list[float] = []
    for l1_comm_id in set(layer1_comms.values()):
        l0_children = [n for n, l in layer_assignments.items() if l == 0 and layer1_comms.get(n, -1) == l1_comm_id]
        if len(l0_children) > 1:
            subgraph = G.subgraph(l0_children)
            consistency_scores.append(float(nx.density(subgraph)))
    return float(np.mean(consistency_scores)) if consistency_scores else 0.0


def cross_branch_diversity(hierarchies: Mapping[str, Mapping[Any, int]], layer_name: str = "layer_1"):
    """
    NEW METRIC: Different branches at same level should be distinct
    High diversity = communities at same level cover different topics
    """
    if layer_name not in hierarchies:
        return 0.0
    communities = hierarchies[layer_name]
    comm_ids = list(set(communities.values()))
    if len(comm_ids) < 2:
        return 0.0
    diversity_scores: list[float] = []
    for i in range(len(comm_ids)):
        for j in range(i + 1, len(comm_ids)):
            nodes_i = {n for n, c in communities.items() if c == comm_ids[i]}
            nodes_j = {n for n, c in communities.items() if c == comm_ids[j]}
            union = nodes_i | nodes_j
            if not union:
                continue
            overlap = len(nodes_i & nodes_j) / len(union)
            diversity_scores.append(1.0 - overlap)
    return float(np.mean(diversity_scores)) if diversity_scores else 0.0


def hierarchical_balanced_objective(
    G: nx.Graph,
    hierarchies: Mapping[str, Mapping[Any, int]],
    layer_assignments: Mapping[Any, int],
    alpha: float = 0.4,
    beta: float = 0.3,
    gamma: float = 0.3,
):
    """
    YOUR NOVEL OBJECTIVE FUNCTION FOR HIERARCHICAL COMMUNITIES
    Returns dict with total_score and components
    """
    cohesion_scores = hierarchical_cohesion(G, hierarchies, layer_assignments)
    avg_cohesion = float(np.mean(list(cohesion_scores.values()))) if cohesion_scores else 0.0
    consistency = vertical_consistency(G, hierarchies, layer_assignments)
    diversity = cross_branch_diversity(hierarchies, layer_name="layer_1")
    total_score = alpha * avg_cohesion + beta * consistency + gamma * diversity
    return {
        "total_score": float(total_score),
        "cohesion": float(avg_cohesion),
        "consistency": float(consistency),
        "diversity": float(diversity),
        "cohesion_per_layer": cohesion_scores,
    }

### Day 4-5: Experiments and visualizations

We compare all hierarchical algorithms using the balanced objective under a parameter sweep over α, β, γ ∈ {0.2, 0.4, 0.6} with α + β + γ = 1. We report the best configuration and visualize trade-offs.

In [ ]:
# Experiments: compare hierarchical algorithms and visualize results
# - Parameter sweep over alpha, beta, gamma
# - Compute objective for each algorithm
# - Visualize scores and best configuration

from typing import Callable, Mapping, Any


def compare_hierarchical_algorithms(G: nx.Graph, layer_assignments: Mapping[Any, int]):
    results = []

    algorithms: dict[str, Callable[[], Mapping[str, Mapping[Any, int]]]] = {
        "Hierarchical_Leiden": lambda: hierarchical_leiden_multilevel(G, layer_assignments),
        "Hierarchical_Louvain": lambda: hierarchical_louvain(G),
        "Agglomerative_Clustering": lambda: agglomerative_clustering_on_embeddings(G),
    }

    grid = []
    for alpha in [0.2, 0.4, 0.6]:
        for beta in [0.2, 0.4, 0.6]:
            gamma = round(1 - alpha - beta, 2)
            if gamma < 0:
                continue
            grid.append((alpha, beta, gamma))

    all_rows = []
    for algo_name, algo_func in algorithms.items():
        try:
            hierarchies = algo_func()
        except Exception as e:
            print(f"Warning: {algo_name} failed: {e}")
            continue
        for alpha, beta, gamma in grid:
            scores = hierarchical_balanced_objective(
                G, hierarchies, layer_assignments, alpha=alpha, beta=beta, gamma=gamma
            )
            row = {
                "algorithm": algo_name,
                "alpha": alpha,
                "beta": beta,
                "gamma": gamma,
                **{k: v for k, v in scores.items() if k != "cohesion_per_layer"},
                # flatten cohesion per layer
                **{f"{k}": v for k, v in scores.get("cohesion_per_layer", {}).items()},
            }
            all_rows.append(row)
    if not all_rows:
        return pd.DataFrame()
    return pd.DataFrame(all_rows)


# Run experiments
results_df = compare_hierarchical_algorithms(G_medical, layer_map)
if results_df.empty:
    print("No results were produced. Check algorithm dependencies.")
else:
    best_idx = results_df["total_score"].idxmax()
    best_row = results_df.loc[best_idx]
    print(
        f"Best algorithm: {best_row['algorithm']}\n"
        f"Best parameters: α={best_row['alpha']}, β={best_row['beta']}, γ={best_row['gamma']}\n"
        f"Score breakdown: Cohesion={best_row['cohesion']:.3f}, "
        f"Consistency={best_row['consistency']:.3f}, Diversity={best_row['diversity']:.3f}"
    )

    # Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=results_df, x="algorithm", y="total_score", ax=axes[0])
    axes[0].set_title("Total Score by Algorithm")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Score")

    sns.scatterplot(
        data=results_df,
        x="cohesion",
        y="diversity",
        hue="algorithm",
        size="consistency",
        sizes=(20, 200),
        ax=axes[1],
    )
    axes[1].set_title("Trade-offs: Cohesion vs Diversity (size=Consistency)")
    plt.tight_layout()
    plt.show()

### Day 6: Hierarchical Graph RAG simulation and evaluation

We simulate a coarse-to-fine retrieval pipeline: map a query to Layer 2, drill into Layer 1, then collect Layer 0 entities. We then compute precision/recall over a small set of domain queries.

In [ ]:
# RAG simulation: coarse-to-fine retrieval over learned hierarchies
# - Simple keyword heuristics to map query to layer-2 cluster
# - Drill down into layer-1 then gather layer-0 entities
# - Evaluate precision/recall on a few test queries

from collections import defaultdict
from typing import Sequence, Tuple, Mapping, Any


def simulate_hierarchical_rag_retrieval(G: nx.Graph, hierarchies: Mapping[str, Mapping[Any, int]], layer_assignments: Mapping[Any, int], query: str):
    # Coarse (Layer 2) selection via keyword heuristics
    terms = query.lower().split()
    l2 = hierarchies.get("layer_2", {})
    if not l2:
        return []

    # Score each layer 2 community by matching node names
    comm_to_nodes = defaultdict(list)
    for n, c in l2.items():
        comm_to_nodes[c].append(n)

    def score_comm(nodes):
        score = 0
        for n in nodes:
            name = str(n).lower()
            if any(t in name for t in terms):
                score += 1
        return score

    best_l2 = max(comm_to_nodes.items(), key=lambda kv: score_comm(kv[1]))[0]
    l2_nodes = set(comm_to_nodes[best_l2])

    # Drill to Layer 1: select concepts attached to chosen Layer 2 nodes
    l1 = hierarchies.get("layer_1", {})
    l1_comm_to_nodes = defaultdict(list)
    for n, c in l1.items():
        l1_comm_to_nodes[c].append(n)

    # Keep only l1 nodes connected to chosen l2 category (via edges or name prefix heuristic)
    candidate_l1 = []
    for nodes in l1_comm_to_nodes.values():
        kept = []
        for n in nodes:
            # heuristic by KG parent edge
            parents = [p for p in G.predecessors(n)] if n in G else []
            if set(parents) & l2_nodes:
                kept.append(n)
            elif any(str(p) in str(n) for p in l2_nodes):
                kept.append(n)
        if kept:
            candidate_l1.extend(kept)

    # Collect Layer 0 entities that descend from candidate Layer 1 nodes
    l0 = hierarchies.get("layer_0", {})
    retrieved = []
    for n in l0:
        if layer_assignments.get(n, -1) != 0:
            continue
        parents = [p for p in G.predecessors(n)] if n in G else []
        if set(parents) & set(candidate_l1):
            retrieved.append(n)
    return retrieved


def evaluate_rag_with_hierarchies(G: nx.Graph, hierarchies_dict: Mapping[str, Mapping[str, Mapping[Any, int]]], test_queries: Sequence[Tuple[str, Sequence[str]]]):
    results: dict[str, dict[str, float]] = {}
    for algo_name, hierarchies in hierarchies_dict.items():
        precisions, recalls = [], []
        for query, gt in test_queries:
            retrieved = simulate_hierarchical_rag_retrieval(G, hierarchies, layer_map, query)
            retrieved_set = set(retrieved)
            gt_set = set(gt)
            if len(retrieved_set) == 0:
                precision = 0.0
            else:
                precision = len(retrieved_set & gt_set) / len(retrieved_set)
            recall = len(retrieved_set & gt_set) / len(gt_set) if gt_set else 0.0
            precisions.append(precision)
            recalls.append(recall)
        avg_p = float(np.mean(precisions)) if precisions else 0.0
        avg_r = float(np.mean(recalls)) if recalls else 0.0
        f1 = (2 * avg_p * avg_r / (avg_p + avg_r)) if (avg_p + avg_r) > 0 else 0.0
        results[algo_name] = {"avg_precision": avg_p, "avg_recall": avg_r, "f1": f1}
    return results


# Prepare hierarchies from algorithms (reuse from experiments if available)
try:
    leiden_hierarchies = hierarchical_leiden_multilevel(G_medical, layer_map)
except Exception as e:
    print("Leiden hierarchies failed:", e)
    leiden_hierarchies = {}

try:
    louvain_hierarchies = hierarchical_louvain(G_medical)
except Exception as e:
    print("Louvain hierarchies failed:", e)
    louvain_hierarchies = {}

# Test queries for diabetes domain
test_queries = [
    (
        "What medications treat Type 2 diabetes?",
        ["Diabetes_Type_2_Metformin", "Diabetes_Type_2_GLP1_Agonists"],
    ),
    (
        "What are complications of diabetes?",
        ["Diabetic_Complications_Retinopathy", "Diabetic_Complications_Nephropathy"],
    ),
]

rag_results = evaluate_rag_with_hierarchies(
    G_medical,
    {"Leiden": leiden_hierarchies, "Louvain": louvain_hierarchies},
    test_queries,
)
rag_results

### Day 7: Paper outline

We capture the high-level paper outline and key contributions as a reference for writing.

#### Title

Hierarchical Community Detection with Balanced Cohesion-Consistency-Diversity Objectives for Multi-Layer Knowledge Graphs in Medical RAG Systems

#### Key Contributions
- Novel objective function for hierarchical communities: α·Cohesion + β·Vertical_Consistency + γ·Cross_Branch_Diversity
- Systematic comparison of hierarchical algorithms (Leiden, Louvain, Agglomerative) on medical KG
- Empirical demonstration that balanced hierarchies improve RAG retrieval performance

#### Outline
- Introduction
  - Hierarchical RAG needs coarse-to-fine community structures; flat methods are inadequate
  - Propose hierarchical objective function
- Related Work
  - Hierarchical community detection, medical knowledge graphs, Graph RAG systems
- Problem Formulation
  - Define hierarchical KG structure (Layer 0/1/2)
  - Objective components: Horizontal cohesion, Vertical consistency, Cross-branch diversity
  - Combined objective function
- Hierarchical Algorithms
  - Hierarchical Leiden, Louvain dendrogram, Agglomerative on embeddings
- Experiments
  - Dataset: Synthetic diabetes hierarchical KG; Baselines: 3 algorithms
  - Parameter sweep: α, β, γ ∈ {0.2, 0.4, 0.6}
  - Results: best configuration table, trade-off plots, hierarchy visualization
- Application to Hierarchical RAG
  - Architecture, retrieval improvement with balanced hierarchies, example walkthrough
- Conclusion
  - Findings, implications, and directions for larger KGs and production integration